In [4]:
from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder

import numpy as np
import chromadb
import anthropic
from dotenv import load_dotenv


In [5]:
import os
key = os.environ.get("ANTHROPIC_API_KEY", "")
print("有讀到金鑰" if key.startswith("sk-ant-") else "沒讀到金鑰，檢查 .env")

沒讀到金鑰，檢查 .env


In [6]:
def load(folder: str) -> list[dict]:
    docs = []
    for p in Path(folder).glob("*.md"):
        text = p.read_text(encoding="utf-8")
        docs.append({"text": text, "source": p.name})
    return docs

In [7]:
docs = load("data")
lines = docs[0]["text"].splitlines()

In [8]:
def split(docs: list[dict]) -> list[dict]:
    chunks = []
    for doc in docs:
        section = ""
        for line in doc["text"].splitlines():
            line = line.strip()
            if line.startswith("## "):
                section = line[3:]
            elif line and not line.startswith("#") and not line.startswith(">"):
                chunks.append({
                    "text": f"【{section}】{line}",
                    "source": doc["source"],
                    "section": section,
                })
    return chunks

In [9]:
chunks = split(docs)
print(f"共 {len(chunks)} 塊")
for c in chunks[:4]:
    print(c)

共 62 塊
{'text': '【店家基本資訊】茶語時光是一家位於台北市大安區的手搖飲料店，地址為台北市大安區和平東路二段 88 號。', 'source': 'tea_shop_kb.md', 'section': '店家基本資訊'}
{'text': '【店家基本資訊】門市電話為 02-2700-1234，官方 LINE 帳號為 @teatime88。', 'source': 'tea_shop_kb.md', 'section': '店家基本資訊'}
{'text': '【菜單總覽】本店飲品共分為三大系列，以下為全部品項。', 'source': 'tea_shop_kb.md', 'section': '菜單總覽'}
{'text': '【菜單總覽】經典茶飲：茉莉綠茶、四季春青茶、阿薩姆紅茶、凍頂烏龍茶、蕎麥茶。', 'source': 'tea_shop_kb.md', 'section': '菜單總覽'}


In [10]:
model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

In [11]:
def embed(texts: list[str]) -> np.ndarray:
    return model.encode(texts,
                        normalize_embeddings = True,
                        show_progress_bar=False)

In [12]:
texts = [c["text"] for c in chunks]
vectors = embed(texts)
print(len(texts), vectors.shape)

62 (62, 512)


In [13]:
client = chromadb.PersistentClient(path= "./db")
# client = chromadb.EphemeralClient()
# client.delete_collection("tea_shop")
col = client.get_or_create_collection("tea_shop")
print(col.count())

62


In [14]:
def store(chunks: list[dict], vectors: np.ndarray) -> None:
    col.upsert(
        ids=[f"chunk-{i}" for i in range(len(chunks))],
        documents=[c["text"] for c in chunks],
        embeddings=vectors.tolist(),
        metadatas=[{"source": c["source"], "section": c["section"]} for c in chunks],
    )

In [15]:
def build_index(folder: str = ".") -> None:
    """重建整個索引：刪掉舊的 collection，重新 load → split → embed → store"""
    global col
    try:
        chromadb.delete_collection("tea_shop")
    except Exception:
        pass
    col = chromadb.get_or_create_collection("tea_shop")

    chunks = split(load(folder))
    store(chunks, embed([c["text"] for c in chunks]))
    print(f"索引完成，共 {col.count()} 塊")

In [16]:
store(chunks, vectors)
print(col.count())
# print(col.get(ids=["chunk-42"], include=["documents", "metadatas"]))

62


In [17]:
q = embed(["孕婦可以喝什麼？"])
res = col.query(query_embeddings=q.tolist(), n_results=3)
print(res["documents"])
print(res["metadatas"])
print(res["distances"])

[['【咖啡因說明】孕婦或對咖啡因敏感的顧客，建議選擇蕎麥茶或鮮果飲。', '【加料選項】每杯飲料最多可加三種配料。', '【奶茶與拿鐵系列】鮮奶茶：中杯 60 元，大杯 70 元。使用 100% 林鳳營鮮乳，不含奶精。']]
[[{'source': 'tea_shop_kb.md', 'section': '咖啡因說明'}, {'section': '加料選項', 'source': 'tea_shop_kb.md'}, {'section': '奶茶與拿鐵系列', 'source': 'tea_shop_kb.md'}]]
[[0.7703853249549866, 1.0296642780303955, 1.0343759059906006]]


In [18]:
d = res["distances"][0][0]
print(f"距離 {d:.3f} → 相似度 {1 - d/2:.3f}")

距離 0.770 → 相似度 0.615


In [19]:
def retrieve(question: str, top_k: int = 5) -> list[dict]:
    q = embed([question])
    res = col.query(query_embeddings=q.tolist(), n_results=top_k)
    results = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        results.append({
            "text": doc,
            "source": meta["source"],
            "section": meta["section"],
            "score": 1 - dist / 2,
        })
    return results

In [20]:
for q in ["孕婦可以喝什麼？", "刷卡有最低消費嗎？", "芒果冰沙可以做熱的嗎？"]:
    print(f"\n問題：{q}")
    for r in retrieve(q, top_k=5):
        print(f"  {r['score']:.3f}  {r['text']}")


問題：孕婦可以喝什麼？
  0.615  【咖啡因說明】孕婦或對咖啡因敏感的顧客，建議選擇蕎麥茶或鮮果飲。
  0.485  【加料選項】每杯飲料最多可加三種配料。
  0.483  【奶茶與拿鐵系列】鮮奶茶：中杯 60 元，大杯 70 元。使用 100% 林鳳營鮮乳，不含奶精。
  0.482  【咖啡因說明】可可鮮奶含有微量咖啡因，大杯約 10 毫克，對咖啡因敏感者請斟酌飲用。
  0.472  【奶茶與拿鐵系列】可可鮮奶：中杯 65 元，大杯 75 元。比利時可可粉加鮮奶，不含茶。

問題：刷卡有最低消費嗎？
  0.606  【付款方式】信用卡僅接受單筆消費 100 元以上。
  0.539  【會員制度】每消費 50 元累積 1 點，集滿 10 點可兌換任一中杯飲品一杯。
  0.507  【付款方式】門市接受現金、LINE Pay、街口支付、悠遊卡與信用卡。
  0.503  【菜單總覽】本店目前沒有販售咖啡。
  0.492  【營業時間】每月最後一個週二為店休日，當天不營業。

問題：芒果冰沙可以做熱的嗎？
  0.709  【甜度與冰塊調整】冰沙類飲品無法做熱飲或溫飲。
  0.665  【甜度與冰塊調整】芒果冰沙與柳橙鮮果飲無法調整為無糖，最低為微糖。
  0.631  【水果茶系列】芒果冰沙：大杯 75 元，僅提供大杯。季節限定，每年 5 月至 9 月販售。
  0.568  【菜單總覽】水果茶與鮮果飲：百香綠茶、檸檬紅茶、柳橙鮮果飲、芒果冰沙（季節限定）。
  0.564  【甜度與冰塊調整】冰塊可選擇：正常冰、少冰、微冰、去冰、溫、熱。


In [21]:
# reranker = CrossEncoder("BAAI/bge-reranker-base")
#
# def rerank(question: str, chunks: list[dict], top_k: int = 5) -> list[dict]:
#     scores = reranker.predict([(question, c["text"]) for c in chunks])
#     ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
#     return [c for c, s in ranked[:top_k]]

一個好的 RAG 提示詞有 4 個部分

① 角色：你是誰、在做什麼
② 規則：只能根據資料回答、找不到怎麼辦、要不要附出處
③ 資料：retrieve() 找到的內容，加上編號
④ 問題：使用者的提問

In [22]:
def build_prompt(question: str, chunks: list[dict]) -> str:
    """把問題和檢索到的資料組成提示詞"""
    lines = [f"[{i}] {c['text']}" for i, c in enumerate(chunks, start=1)]
    context = "\n".join(lines)

    return f"""你是「茶語時光」手搖飲店的客服助理。

規則：
1. 只能根據 <資料> 中的內容回答，不要自己推測或補充。
2. 如果資料中沒有答案，請回答「抱歉，這個問題我無法回答，請洽門市 02-2700-1234。」
3. 回答要親切，使用繁體中文。
4. 如果資料中有多個符合條件的選項，請全部列出，不要只挑其中幾個。
5. 在回答最後標出參考的資料編號，例如：（參考：[1][3]）
6. 不要使用 Markdown 格式

<資料>
{context}
</資料>

<問題>
{question}
</問題>"""

In [23]:
q = "芒果冰沙可以做熱的嗎？"
print(build_prompt(q, retrieve(q)))

你是「茶語時光」手搖飲店的客服助理。

規則：
1. 只能根據 <資料> 中的內容回答，不要自己推測或補充。
2. 如果資料中沒有答案，請回答「抱歉，這個問題我無法回答，請洽門市 02-2700-1234。」
3. 回答要親切，使用繁體中文。
4. 如果資料中有多個符合條件的選項，請全部列出，不要只挑其中幾個。
5. 在回答最後標出參考的資料編號，例如：（參考：[1][3]）
6. 不要使用 Markdown 格式

<資料>
[1] 【甜度與冰塊調整】冰沙類飲品無法做熱飲或溫飲。
[2] 【甜度與冰塊調整】芒果冰沙與柳橙鮮果飲無法調整為無糖，最低為微糖。
[3] 【水果茶系列】芒果冰沙：大杯 75 元，僅提供大杯。季節限定，每年 5 月至 9 月販售。
[4] 【菜單總覽】水果茶與鮮果飲：百香綠茶、檸檬紅茶、柳橙鮮果飲、芒果冰沙（季節限定）。
[5] 【甜度與冰塊調整】冰塊可選擇：正常冰、少冰、微冰、去冰、溫、熱。
</資料>

<問題>
芒果冰沙可以做熱的嗎？
</問題>


In [24]:
load_dotenv()
llm = anthropic.Anthropic()


In [25]:
def generate(prompt: str) -> str:
    msg = llm.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in msg.content if b.type == "text")

In [26]:
def ask(question: str, top_k: int = 3) -> str:
    chunks = retrieve(question, top_k)
    prompt = build_prompt(question, chunks)
    return generate(prompt)

In [27]:
questions = [
    "芒果冰沙可以做熱的嗎？",
    "刷卡有最低消費嗎？",
    "孕婦可以喝什麼？",
    "你們有賣咖啡嗎？",
]
for q in questions:
    print(f"🙋 {q}")
    print(f"🤖 {ask(q)}\n")

🙋 芒果冰沙可以做熱的嗎？
🤖 不好意思，芒果冰沙屬於冰沙類飲品，無法做熱飲或溫飲喔。如果想喝熱的，歡迎改選其他飲品，也可以洽詢門市了解更多選擇。（參考：[1]）

🙋 刷卡有最低消費嗎？
🤖 有的，使用信用卡付款需要單筆消費滿 100 元以上喔。如果消費未滿 100 元，也可以改用現金、LINE Pay、街口支付或悠遊卡付款。（參考：[1][3]）

🙋 孕婦可以喝什麼？
🤖 您好，孕婦或對咖啡因敏感的顧客，建議選擇蕎麥茶或鮮果飲喔！如果想了解這兩類飲品的品項或價格，歡迎洽詢門市，我們很樂意為您說明。（參考：[1]）

🙋 你們有賣咖啡嗎？
🤖 您好～很抱歉，本店目前沒有販售咖啡，咖啡系列也暫無販售。歡迎您參考我們的其他飲品喔！😊（參考：[1][2]）



In [28]:
retrieve("孕婦可以喝什麼？", top_k=5)

[{'text': '【咖啡因說明】孕婦或對咖啡因敏感的顧客，建議選擇蕎麥茶或鮮果飲。',
  'source': 'tea_shop_kb.md',
  'section': '咖啡因說明',
  'score': 0.6148073375225067},
 {'text': '【加料選項】每杯飲料最多可加三種配料。',
  'source': 'tea_shop_kb.md',
  'section': '加料選項',
  'score': 0.48516786098480225},
 {'text': '【奶茶與拿鐵系列】鮮奶茶：中杯 60 元，大杯 70 元。使用 100% 林鳳營鮮乳，不含奶精。',
  'source': 'tea_shop_kb.md',
  'section': '奶茶與拿鐵系列',
  'score': 0.4828120470046997},
 {'text': '【咖啡因說明】可可鮮奶含有微量咖啡因，大杯約 10 毫克，對咖啡因敏感者請斟酌飲用。',
  'source': 'tea_shop_kb.md',
  'section': '咖啡因說明',
  'score': 0.48232877254486084},
 {'text': '【奶茶與拿鐵系列】可可鮮奶：中杯 65 元，大杯 75 元。比利時可可粉加鮮奶，不含茶。',
  'source': 'tea_shop_kb.md',
  'section': '奶茶與拿鐵系列',
  'score': 0.4717073440551758}]

In [42]:
def retrieve_with_sections(question: str, top_k: int = 3) -> list[dict]:
    hits = retrieve(question, top_k)
    sections = list(dict.fromkeys(h["section"] for h in hits))
    results = []
    for sec in sections:
        res = col.get(where={"section": sec}, include=["documents", "metadatas"])
        rows = sorted(
            zip(res["ids"], res["documents"], res["metadatas"]),
            key=lambda r: int(r[0].split("-")[1]),     # "chunk-23" → 23，依原文順序排
        )
        for _id, doc, meta in rows:
            results.append({"text": doc, "source": meta["source"], "section": meta["section"]})
    return results

In [43]:
for c in retrieve_with_sections("孕婦可以喝什麼？", top_k=3):
    print(c["text"])

【咖啡因說明】茉莉綠茶、四季春青茶、阿薩姆紅茶、凍頂烏龍茶皆含有咖啡因，大杯約含 80 至 120 毫克。
【咖啡因說明】所有奶茶與以茶為基底的水果茶也含有咖啡因。
【咖啡因說明】以下飲品不含咖啡因：蕎麥茶、黑糖珍珠鮮奶、柳橙鮮果飲、芒果冰沙。
【咖啡因說明】可可鮮奶含有微量咖啡因，大杯約 10 毫克，對咖啡因敏感者請斟酌飲用。
【咖啡因說明】孕婦或對咖啡因敏感的顧客，建議選擇蕎麥茶或鮮果飲。
【加料選項】每份加料價格如下：珍珠 10 元、椰果 10 元、仙草凍 10 元、布丁 15 元、芋圓 15 元、奶蓋 20 元。
【加料選項】每杯飲料最多可加三種配料。
【加料選項】奶蓋含有乳製品與鹽，口味偏鹹甜。
【奶茶與拿鐵系列】珍珠奶茶：中杯 55 元，大杯 65 元。阿薩姆紅茶搭配奶精與黑糖珍珠。
【奶茶與拿鐵系列】鮮奶茶：中杯 60 元，大杯 70 元。使用 100% 林鳳營鮮乳，不含奶精。
【奶茶與拿鐵系列】烏龍拿鐵：中杯 65 元，大杯 75 元。凍頂烏龍加鮮奶。
【奶茶與拿鐵系列】黑糖珍珠鮮奶：中杯 70 元，大杯 80 元。不含茶，只有鮮奶與黑糖珍珠。
【奶茶與拿鐵系列】可可鮮奶：中杯 65 元，大杯 75 元。比利時可可粉加鮮奶，不含茶。


In [31]:
def ask(question: str, top_k: int = 3) -> str:
    chunks = retrieve_with_sections(question, top_k)
    prompt = build_prompt(question, chunks)
    return generate(prompt)

print(ask("孕婦可以喝什麼？"))

您好～針對孕婦或對咖啡因敏感的顧客，我們建議選擇蕎麥茶或鮮果飲，目前菜單中的鮮果飲是柳橙鮮果飲。

另外，以下飲品也都不含咖啡因：
1. 蕎麥茶
2. 黑糖珍珠鮮奶
3. 柳橙鮮果飲
4. 芒果冰沙

提醒您，可可鮮奶含有微量咖啡因（大杯約 10 毫克），茶類、奶茶和以茶為基底的水果茶也都含有咖啡因，點餐時請多留意喔！

（參考：[2][3][4][5]）


In [32]:
TEST_SET = [
    ("孕婦可以喝什麼？",           "以下飲品不含咖啡因"),
    ("芒果冰沙可以做熱的嗎？",     "冰沙類飲品無法做熱飲"),
    ("我對蛋過敏，有什麼要注意？", "布丁含有蛋"),
    ("黑糖珍珠鮮奶大杯多少錢？",   "黑糖珍珠鮮奶：中杯 70 元，大杯 80 元"),
    ("刷卡有最低消費嗎？",         "信用卡僅接受單筆消費 100 元以上"),
    ("點數多久會過期？",           "點數有效期限"),
    ("你們平日幾點關門？",         "晚上 9:30"),
    ("過年有開嗎？",               "大年初一至初三公休"),
    ("可以帶自己的杯子嗎？",       "自備環保杯可折抵 5 元"),
    ("公司要訂 30 杯怎麼訂？",     "20 杯以上為大量訂購"),
    ("飲料做錯了怎麼辦？",         "免費重新製作"),
    ("黑糖珍珠鮮奶可以無糖嗎？",   "選擇無糖時仍會有黑糖的甜味"),
]

In [44]:
def evaluate(retriever, test_set: list[tuple[str, str]], top_k: int = 3) -> None:
    hits = 0
    total_chunks = 0
    for question, must_have in test_set:
        chunks = retriever(question, top_k)
        found = any(must_have in c["text"] for c in chunks)
        hits += found
        total_chunks += len(chunks)
        if not found:
            print(f"  ✗ {question}")
    print(f"  命中率 {hits}/{len(test_set)}，平均帶入 {total_chunks / len(test_set):.1f} 塊")

In [45]:
for k in [1, 3, 5]:
    print(f"retrieve, top_k={k}")
    evaluate(retrieve, TEST_SET, top_k=k)

print("retrieve_with_sections, top_k=3")
evaluate(retrieve_with_sections, TEST_SET, top_k=3)

retrieve, top_k=1
  ✗ 孕婦可以喝什麼？
  ✗ 你們平日幾點關門？
  命中率 10/12，平均帶入 1.0 塊
retrieve, top_k=3
  ✗ 孕婦可以喝什麼？
  ✗ 你們平日幾點關門？
  命中率 10/12，平均帶入 3.0 塊
retrieve, top_k=5
  ✗ 孕婦可以喝什麼？
  命中率 11/12，平均帶入 5.0 塊
retrieve_with_sections, top_k=3
  命中率 12/12，平均帶入 8.8 塊


In [46]:
all_meta = col.get(include=["metadatas"])["metadatas"]
SECTIONS = list(dict.fromkeys(m["section"] for m in all_meta))
print(SECTIONS)

['店家基本資訊', '菜單總覽', '營業時間', '經典茶飲菜單', '奶茶與拿鐵系列', '水果茶系列', '咖啡系列', '咖啡因說明', '甜度與冰塊調整', '加料選項', '過敏原資訊', '會員制度', '外送與預訂', '環保與自備杯', '退換與客訴', '付款方式']


In [37]:
def search_kb(query: str) -> str:
    """語意搜尋，重複利用你寫的 retrieve_with_sections"""
    chunks = retrieve_with_sections(query, top_k=3)
    return "\n".join(c["text"] for c in chunks)


def get_section(name: str) -> str:
    """取出某個段落的全部內容"""
    res = col.get(where={"section": name}, include=["documents"])
    if not res["documents"]:
        return f"找不到段落「{name}」。可用的段落有：{'、'.join(SECTIONS)}"
    return "\n".join(res["documents"])


TOOL_FUNCS = {"search_kb": search_kb, "get_section": get_section}

print(get_section("水果茶系列"))

【水果茶系列】百香綠茶：中杯 50 元，大杯 60 元。新鮮百香果搭配茉莉綠茶。
【水果茶系列】檸檬紅茶：中杯 45 元，大杯 55 元。每杯使用半顆新鮮檸檬現榨。
【水果茶系列】柳橙鮮果飲：中杯 60 元，大杯 70 元。現榨柳橙汁，不含茶。
【水果茶系列】芒果冰沙：大杯 75 元，僅提供大杯。季節限定，每年 5 月至 9 月販售。


In [38]:
TOOLS = [
    {
        "name": "search_kb",
        "description": "用語意搜尋茶語時光的知識庫。適合查詢具體問題，例如某個飲料能不能做熱的、刷卡規定、營業時間。",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "要搜尋的問題或關鍵字"},
            },
            "required": ["query"],
        },
    },
    {
        "name": "get_section",
        "description": f"取出知識庫中某個段落的完整內容。適合需要完整清單的問題，例如全部菜單、全部規定。可用的段落：{'、'.join(SECTIONS)}",
        "input_schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "段落名稱，必須是可用段落之一"},
            },
            "required": ["name"],
        },
    },
]

SYSTEM = """你是「茶語時光」手搖飲店的客服助理。

規則：
1. 回答前一定要用工具查詢資料，只能根據查到的內容回答，不要自己推測。
2. 如果查不到答案，請回答「抱歉，這個問題我無法回答，請洽門市 02-2700-1234。」
3. 回答要親切，使用繁體中文。
4. 如果有多個符合條件的選項，請全部列出。"""

In [39]:
msg = llm.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=2000,
    system=SYSTEM,
    tools=TOOLS,
    messages=[{"role": "user", "content": "你們有什麼飲料？"}],
)
print("stop_reason:", msg.stop_reason)
for b in msg.content:
    if b.type == "tool_use":
        print(f"🔧 {b.name}({b.input})")
    else:
        print(b.type)

stop_reason: tool_use
🔧 get_section({'name': '菜單總覽'})


In [40]:
def agent_ask(question: str, max_steps: int = 6) -> str:
    messages = [{"role": "user", "content": question}]

    for step in range(max_steps):
        msg = llm.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=2000,
            system=SYSTEM,
            tools=TOOLS,
            messages=messages,
        )

        if msg.stop_reason != "tool_use":
            return "".join(b.text for b in msg.content if b.type == "text")

        messages.append({"role": "assistant", "content": msg.content})

        results = []
        for b in msg.content:
            if b.type == "tool_use":
                print(f"  🔧 {b.name}({b.input})")
                output = TOOL_FUNCS[b.name](**b.input)
                results.append({
                    "type": "tool_result",
                    "tool_use_id": b.id,
                    "content": output,
                })

        messages.append({"role": "user", "content": results})

    return "抱歉，這個問題查詢太多次了，請洽門市 02-2700-1234。"

In [41]:
for q in ["你們有什麼飲料？", "芒果冰沙可以做熱的嗎？", "孕婦可以喝什麼？"]:
    print(f"\n🙋 {q}")
    print(f"🤖 {agent_ask(q)}")


🙋 你們有什麼飲料？
  🔧 get_section({'name': '菜單總覽'})
  🔧 get_section({'name': '經典茶飲菜單'})
  🔧 get_section({'name': '奶茶與拿鐵系列'})
  🔧 get_section({'name': '水果茶系列'})
  🔧 get_section({'name': '咖啡系列'})
🤖 我們的飲品分成三大系列，價格如下：

**🍵 經典茶飲**（中杯／大杯）
- 茉莉綠茶：35／40 元
- 四季春青茶：35／40 元
- 阿薩姆紅茶：35／40 元
- 凍頂烏龍茶：40／45 元
- 蕎麥茶（不含茶葉）：35／40 元

**🥛 奶茶與拿鐵**
- 珍珠奶茶：55／65 元
- 鮮奶茶（100% 林鳳營鮮乳，不含奶精）：60／70 元
- 烏龍拿鐵：65／75 元
- 黑糖珍珠鮮奶（不含茶）：70／80 元
- 可可鮮奶（不含茶）：65／75 元

**🍋 水果茶與鮮果飲**
- 百香綠茶：50／60 元
- 檸檬紅茶：45／55 元
- 柳橙鮮果飲（不含茶）：60／70 元
- 芒果冰沙（季節限定，5～9 月販售）：僅大杯 75 元

目前店內沒有販售咖啡，敬請見諒。如果想了解加料、甜度冰塊或咖啡因含量，也可以隨時問我喔！😊

🙋 芒果冰沙可以做熱的嗎？
  🔧 search_kb({'query': '芒果冰沙可以做熱的嗎'})
  🔧 get_section({'name': '水果茶系列'})
🤖 芒果冰沙沒辦法做熱的喔。冰沙類飲品無法做熱飲或溫飲。

芒果冰沙是季節限定,每年 5 月到 9 月販售,只有大杯,75 元。甜度最低可調到微糖,不能選無糖。

如果你想喝熱飲,可以看看其他品項。需要的話,我可以幫你查哪些飲料可以做熱的。😊

🙋 孕婦可以喝什麼？
  🔧 search_kb({'query': '孕婦可以喝什麼'})
  🔧 get_section({'name': '咖啡因說明'})
🤖 您好!孕婦或對咖啡因敏感的顧客,店裡建議選擇不含咖啡因的飲品:

**不含咖啡因的飲品**
- 蕎麥茶
- 黑糖珍珠鮮奶(不含茶,只有鮮奶與黑糖珍珠)
- 柳橙鮮果飲
- 芒果冰沙

**店內建議**:蕎麥茶或鮮果飲。

**提醒**
- 可可鮮奶含微量咖啡因,大杯約 